# 0) Terminal (only developer):

**Эта графа только для разработчика, проверяющему не нужно выполнять ее.**

**Все действия выполняются в 0.3)**
```
В этой графе будут скрипты для связи colab и github
Загрузка и преобразование сырых данных в данные .csv
Сохранение их на google drive для дальнейшего использования
```

In [1]:
# подключимся к своему google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0.1) Связываем colab и github для ведения версий.


In [2]:
%%writefile create_conf.sh
#!/bin/bash

# Настройки для связи colab и github.

# Создадим файл config и запишем следующие настройки
c_path="/root/.ssh/config"
touch $c_path
echo "Host github.com" > $c_path
echo "  HostName github.com" >> $c_path
echo "  User git" >> $c_path
echo "  IdentityFile ~/.ssh/id_rsa" >> $c_path
echo "  IdentitiesOnly yes" >> $c_path

# Выполним команду для проверки
ssh github.com
# еще
# ssh -T git@github.com

# Добавляем в конфиг git
git config --global user.email "yury@no.name"
git config --global user.name "Yury"
# для принудительного использования Git SSH вместо HTTPS
git config --global url."git@github.com:".insteadOf "https://github.com/"


Writing create_conf.sh


In [3]:
# делаем исполняемым
!chmod +x create_conf.sh

In [4]:
%%writefile create_relations.sh
#!/bin/bash

# Связь между colab и github.
# Создаем ключи, прописываем настройки.
# Параметр $1 - путь до директории, куда сохраним наши ключи

if [ -z "$1" ]; then
    echo "Ошибка: укажите путь для сохранения ключей"
    exit 1
fi

# Сгенерируем ключ SSH.
# Путь оставляем по умолчанию; можно использовать парольное слово для ключа
# -t rsa — тип ключа RSA;
# -b 4096 — длина ключа в битах;
# -C "colab-github" — комментарий
# Создается директори /root/.ssh/ и содержит id_rsa и id_rsa.pub
ssh-keygen -t rsa -b 4096 -C "colab-github"

# Полученный ключ /root/.ssh/id_rsa.pub указываем в настройках SSH на github
echo ""
cat /root/.ssh/id_rsa.pub
echo ""

read -p "Добавьте ключ на github в секцию SSH и нажмите Enter..."

# сохраняем наши ключи для дальнейшего использования
cp /root/.ssh/id_rsa $1/
cp /root/.ssh/id_rsa.pub $1/

# настройки
./create_conf.sh

Writing create_relations.sh


In [5]:
# делаем исполняемым
!chmod +x create_relations.sh

In [6]:
%%writefile connect_relations.sh
#!/bin/bash

# Устанавливаем связь colab и github, используя имеющиеся ключи
# Параметр $1 - путь до директории, где наши ключи

if [ -z "$1" ]; then
    echo "Ошибка: укажите путь с ключами"
    exit 1
fi

# создадим папку /root/.ssh
mkdir /root/.ssh/

# копируем туда ключи
cp $1/id_rsa /root/.ssh/
cp $1/id_rsa.pub /root/.ssh/

# изменим права, чтоб система не ругалась, что у нас закрытый ключ виден всем
chmod 400 /root/.ssh/id_rsa

# настройки
./create_conf.sh

Writing connect_relations.sh


In [7]:
# делаем исполняемым
!chmod +x connect_relations.sh

In [8]:
!git --version

git version 2.34.1


```shell
# Ключи лежат на google drive:
/content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/rsa_keys
# если ключи уже есть, соединяемся с github
./connect_relations.sh /content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/rsa_keys
# если нет, создаем ключи и соединяемся с github
./create_relations.sh /content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/rsa_keys
```



## 0.2) Загружаем проект с github. Создаем .csv

**Выполняем в терминале:**
```shell
# делаем клон репозитория, помещаем на google drive
git clone https://github.com/YuryRF/tusur_proj.git "/content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/tusur_project"
# если нужно установить удаленный репозиторий
# git remote set-url origin git@github.com:YuryRF/tusur_proj.git
```

**Файлы сырых данных из хэша программы SBPro лежат по адресу:**
```shell
https://drive.google.com/drive/folders/1MnVeILpXZrzQGKc_CytEIwKMs50JVVO7?usp=sharing
```

**Или прямой ссылкой:**
```shell
# создать прямую ссылку по расшареной
https://www.votix.ru/p/sozdat-pryamuyu-ssylku-fajla-google-disk.html
# 604800.zip
https://drive.google.com/uc?export=download&id=1U3wi5ELKdyV2VLqlNxVZzMSlzU1WLUKw
# 86400.zip
https://drive.google.com/uc?export=download&id=1sPWT5N3bW3We5esg4n_Y-bygJWb-iJ82
# 3600.zip
https://drive.google.com/uc?export=download&id=10tcsgw073xO6XeBmrr_5Iptzf5AyjlYI
# 300.zip
https://drive.google.com/uc?export=download&id=1spVBOyoF9W3RjsdaqVjQIlcUErk94JrV
```

**Мы знаем адреса директорий исходных данных, подправим скрипт из репозитория:**
```python
if __name__ == '__main__':
    dir_1w = r"/content/data_source/604800/0.01/Central Standard Time"
    dir_1d = r"/content/data_source/86400/0.01/Central Standard Time"
    dir_1h = r"/content/data_source/3600/0.01/Central Standard Time"
    dir_5m = r"/content/data_source/300/0.01/Central Standard Time/0.0"
    new_path = r"/content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/data/"
    create_bars_dataset("1w", dir_1w, new_path)
    create_bars_dataset("1d", dir_1d, new_path)
    create_bars_dataset("1h", dir_1h, new_path)
    create_bars_dataset("5m", dir_5m, new_path)
```

In [9]:
from pathlib import Path

def create_csv():
  """
  Создаем .csv из исходных файлов, используя ранее написанный скрипт в репозитории
  """
  # проверяем, получили ли мы уже данные
  check_files =  Path('/content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/data/')
  if any(check_files.iterdir()):
    print("Данные уже получены")
    return

  # создадим директорию
  !mkdir data_source

  # Скачиваем и разархивируем
  source_data = {
      "604800.zip": "https://drive.google.com/uc?export=download&id=1U3wi5ELKdyV2VLqlNxVZzMSlzU1WLUKw",
      "86400.zip": "https://drive.google.com/uc?export=download&id=1sPWT5N3bW3We5esg4n_Y-bygJWb-iJ82",
      "3600.zip": "https://drive.google.com/uc?export=download&id=10tcsgw073xO6XeBmrr_5Iptzf5AyjlYI",
      "300.zip": "https://drive.google.com/uc?export=download&id=1spVBOyoF9W3RjsdaqVjQIlcUErk94JrV"
  }
  for key, val in source_data.items():
    # мы могли сразу делать unzip по внутреннему пути:
    # /content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/data_source/
    print("download:", key)
    !wget -q -O data_source/{key} "{val}"
    print("unzip:", key)
    !unzip -q data_source/{key} -d data_source

  # мы предварительно настроили скрипт из репозитория. Выполним его
  !python /content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/tusur_project/create_csv.py


## 0.3) Старт и работа с github:

In [10]:
prompt = """--------------------------------------------
Если нужно работать с gihub, то выполните одно из двух действий:
1) если ключи уже есть:
  ./connect_relations.sh /content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/rsa_keys
2) если ключей нет:
  ./create_relations.sh /content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/rsa_keys
Нажмите Enter для продолжения..."""
p1 = input(prompt)

prompt = """--------------------------------------------
Если это первый запуск скрипта, то нужно сделать клон репозитория:
  git clone https://github.com/YuryRF/tusur_proj.git "/content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/tusur_project"
Нажмите Enter для продолжения..."""
p1 = input(prompt)

prompt = """--------------------------------------------
Если не получили .csv, то исправьте файл:
/content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/tusur_project/create_csv.py
Укажите директории с сырыми данными
Нажмите Enter для продолжения..."""
p1 = input(prompt)

# выполняем эту ф-ию
print("--------------------------------------------\n")
create_csv()

# если мы только клонировали репозиторий и в нем нет файла main.py,
# то скопируем текущий файл в папку с проектом
file_str = '/content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/tusur_project/main.ipynb'
check_file =  Path(file_str)
if check_file.exists():
  print("main.py уже содержится в папке с проектом")
else:
  print("M14.ipynb -> main.ipynb [директория проекта]")
  !cp /content/drive/MyDrive/Colab_Notebooks/AI_2025/M14.ipynb {file_str}

print("--------------------------------------------\n",
      "Для работы с репозиторием github смотрите команды ниже:")

--------------------------------------------
Если нужно работать с gihub, то выполните одно из двух действий:
1) если ключи уже есть:
  ./connect_relations.sh /content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/rsa_keys
2) если ключей нет:
  ./create_relations.sh /content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/rsa_keys
Нажмите Enter для продолжения...
--------------------------------------------
Если это первый запуск скрипта, то нужно сделать клон репозитория:
git clone https://github.com/YuryRF/tusur_proj.git "/content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/tusur_project"
Нажмите Enter для продолжения...
--------------------------------------------
Если не получили .csv, то исправьте файл:
/content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/tusur_project/create_csv.py
Укажите директории с сырыми данными
Нажмите Enter для продолжения...
--------------------------------------------

download: 604800.zip
unzip: 604800.zip
download: 86400.zip
unzip: 86400.zip
download: 3600.zip
unzip



```shell
# Перейти в папку проекта:
cd /content/drive/MyDrive/Colab_Notebooks/AI_2025/M14/tusur_project/
# Проверить статус:
git status
# Добавить файлы для внесения обновлений:
git add .
или
git add some_file
# Закомминить:
git commit -m "комментарий"
# Запушить (внести изменения в репозиторий):
git push origin main
# Получение изменений из удаленного репозитория без их применения к текущим файлам
git fetch
# Посмотреть, что изменилось
git log HEAD..origin/main
# скачивает изменения и сразу объединяет их с вашей текущей локальной веткой (эквивалентно git fetch + git merge)
git pull
```

# 1. Начало работы